In [8]:
!conda install -y -q bffile


Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

# All requested packages already installed.



In [9]:
import os
import sys
from pathlib import Path

import scyjava

scyjava.config.add_classpath("/home/mutterer/Downloads/btnz/ImageJ/plugins/jars/logback-core-1.5.18.jar")
scyjava.config.add_classpath("/home/mutterer/Downloads/btnz/ImageJ/plugins/jars/logback-classic-1.5.18.jar")
scyjava.config.add_classpath("/home/mutterer/Downloads/btnz/ImageJ/plugins/jars/slf4j-api-2.0.17.jar")

from bffile import BioFile
from bffile import BioFile

input_path = "../data/2109017.czi"
        
with BioFile(str(input_path)) as bf:
    print("Bio-Formats version:", BioFile.bioformats_version())
    meta_type = params_dict.get("metadata", "").lower() if "metadata" in params_dict else None
    # Determine series, z, c, t indices from params, default to 0
    def parse_int_param(key, default=0):
        val = params_dict.get(key, default)
        if isinstance(val, bool):
            return int(val)
        try:
            return int(val)
        except Exception:
            return default
    series_idx = parse_int_param("series", 0)
    z_idx = parse_int_param("z", 0)
    c_idx = parse_int_param("c", 0)
    t_idx = parse_int_param("t", 0)
    # display_flag is True if 'display' key is present (regardless of value)
    display_flag = "display" in params_dict
    if meta_type == "ome":
        print("OME metadata:")
        ome = bf.ome_metadata
        print(ome)
        try:
            img = ome.images[series_idx]
            px = img.pixels
            print("SizeX:", px.size_x)
            print("SizeY:", px.size_y)
            print("SizeZ:", px.size_z)
            print("SizeC:", px.size_c)
            print("SizeT:", px.size_t)
            print("Dimension order:", px.dimension_order)
        except Exception as e:
            print(f"Error extracting OME image info: {e}", file=sys.stderr)
    elif meta_type == "core":
        print("Core metadata:")
        try:
            meta = bf.core_metadata(series=series_idx, resolution=0)
            print(meta)
            print("Shape:", getattr(meta, "shape", None))
            print("Dtype:", getattr(meta, "dtype", None))
            print("Dimension order:", getattr(meta, "dimension_order", None))
            print("Is little endian:", getattr(meta, "is_little_endian", None))
        except Exception as e:
            print(f"Error extracting core metadata: {e}", file=sys.stderr)
    elif meta_type == "global":
        print("Global metadata:")
        try:
            global_meta = bf.global_metadata()
            for key, value in global_meta.items():
                print(f"{key}: {value}")
        except Exception as e:
            print(f"Error extracting global metadata: {e}", file=sys.stderr)
    else:
        print("OME metadata:")
        ome = bf.ome_metadata
        print(ome)
        try:
            img = ome.images[series_idx]
            px = img.pixels
            print("SizeX:", px.size_x)
            print("SizeY:", px.size_y)
            print("SizeZ:", px.size_z)
            print("SizeC:", px.size_c)
            print("SizeT:", px.size_t)
            print("Dimension order:", px.dimension_order)
        except Exception as e:
            print(f"Error extracting OME image info: {e}", file=sys.stderr)
        arr = bf.as_array(series=series_idx)
        print("Shape:", arr.shape)
        print("Dtype:", arr.dtype)
    # If display flag is set, read and show the plane
    if display_flag:
        try:
            import matplotlib.pyplot as plt
            # Try bf.read if available, else fallback to as_array
            plane = None
            if hasattr(bf, "read"):
                plane = bf.read(series=series_idx, z=z_idx, c=c_idx, t=t_idx)
            else:
                arr = bf.as_array(series=series_idx)
                # Try to index as (z, c, t, y, x) or (t, c, z, y, x) or similar
                # Use the shape to guess the order
                shape = arr.shape
                # Find the indices for z, c, t
                # Common orders: (t, c, z, y, x), (z, c, t, y, x), (c, z, t, y, x), etc.
                # We'll try to match the shape to the params
                # Assume y, x are always last two dims
                if len(shape) >= 5:
                    # Try to find which axes are which by matching shape to known dimension sizes
                    # Use OME metadata if available
                    try:
                        px = bf.ome_metadata.images[series_idx].pixels
                        sizez, sizec, sizet = px.size_z, px.size_c, px.size_t
                        # Find indices in arr.shape that match these sizes
                        dim_map = {}
                        for idx, dim in enumerate(shape[:-2]):
                            if dim == sizez and "z" not in dim_map:
                                dim_map["z"] = idx
                            elif dim == sizec and "c" not in dim_map:
                                dim_map["c"] = idx
                            elif dim == sizet and "t" not in dim_map:
                                dim_map["t"] = idx
                        # Default to 0 if not found
                        z_axis = dim_map.get("z", 0)
                        c_axis = dim_map.get("c", 1)
                        t_axis = dim_map.get("t", 2)
                        # Build index tuple
                        idxs = [slice(None)] * len(shape)
                        idxs[z_axis] = z_idx
                        idxs[c_axis] = c_idx
                        idxs[t_axis] = t_idx
                        # y, x left as slice(None)
                        plane = arr[tuple(idxs)]
                    except Exception as e:
                        print(f"Error inferring axes for display: {e}", file=sys.stderr)
                        # Fallback: try arr[z, c, t] if possible
                        try:
                            plane = arr[z_idx, c_idx, t_idx]
                        except Exception as e2:
                            print(f"Error extracting plane: {e2}", file=sys.stderr)
                else:
                    print(f"Array shape not suitable for plane extraction: {shape}", file=sys.stderr)
            if plane is not None:
                plt.imshow(plane, cmap="gray")
                plt.title(f"series={series_idx}, z={z_idx}, c={c_idx}, t={t_idx}")
                plt.show()
            else:
                print("Could not extract plane for display.", file=sys.stderr)
        except Exception as e:
            print(f"Error displaying plane: {e}", file=sys.stderr)


TypeError: Class loci.formats.ImageReader is not found